**This is used fot he preprocessing or cleaning of the raw Veritasium youtube video comments.**

In [1]:
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

df = pd.read_csv("Veritasium_youtube_comments_raw.csv")
print(df.shape)
df.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


(14127, 5)


,author,updated_at,like_count,text,public
0,@DanielRenardAnimation,2019-09-19T18:28:21Z,5751,*Russian Cosmonaut spins a wingnut in space:* ...,True
1,@NLTops,2020-03-07T07:17:39Z,5244,Someone should tell the Flat Earthers of the D...,True
2,@aviatordude1961,2021-07-26T02:06:48Z,4005,I thought the reason the Russians kept this a ...,True
3,@alvirahesc7436,2021-04-10T12:21:42Z,3842,"""Babe, come over, im home alone""\n\n""No, babe,...",True
4,@zeekjones1,2019-09-20T05:46:15Z,2814,Over 300 people broke their phone after watchi...,True


1. Drop empty comments

In [3]:
df = df.dropna(subset=['text'])
df = df.drop_duplicates(subset=['text'])
print("After dedup:", df.shape)

After dedup: (13982, 5)


2. Normalize text

In [4]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'&\w+;', '', text)                    # remove HTML entities
    text = re.sub(r'([!?.])\1+', r'\1', text)            # collapse repeated punctuation
    text = re.sub(r'[^a-z0-9\s.,!?\'-]', '', text)       # remove weird characters
    text = re.sub(r'\s+', ' ', text).strip()             # collapse whitespace
    return text

df['text_normalized'] = df['text'].apply(normalize_text)
df[['text','text_normalized']].head()

,text,text_normalized
0,*Russian Cosmonaut spins a wingnut in space:* ...,russian cosmonaut spins a wingnut in space tel...
1,Someone should tell the Flat Earthers of the D...,someone should tell the flat earthers of the d...
2,I thought the reason the Russians kept this a ...,i thought the reason the russians kept this a ...
3,"""Babe, come over, im home alone""\n\n""No, babe,...","babe, come over, im home alone no, babe, im so..."
4,Over 300 people broke their phone after watchi...,over 300 people broke their phone after watchi...


3. Filter out short comments

In [5]:
df = df[df['text_normalized'].str.split().apply(len) >= 3]
print("After length filter:", df.shape)

After length filter: (13366, 6)


4. Tokenize + remove punctuation + stopwords.

In [6]:
stop_words = set(stopwords.words('english'))

def tokenize_clean(text):
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in string.punctuation]
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

df['tokens'] = df['text_normalized'].apply(tokenize_clean)

5. Language filter

In [7]:
!pip install langdetect
from langdetect import detect, LangDetectException

def is_english(text):
    try:
        return detect(text) == 'en'
    except LangDetectException:
        return False

df = df[df['text_normalized'].apply(is_english)]
print(df.shape)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 36.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=b5edbb95a79baef720e86f1fae784e06a48a767981849b6f594ab9c0c500e7d8
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
(12899, 7)


6. Saving cleaned dataset

In [8]:
df = df.reset_index(drop=True)
df['comment_id'] = df.index

df['tokens_str'] = df['tokens'].apply(lambda x: ' '.join(x))

output_cols = ['comment_id','author','updated_at','like_count','text','text_normalized','tokens_str']
df[output_cols].to_csv("Veritasium_youtube_comments_clean.csv", index=False)

print(f"Saved {len(df)} cleaned comments")

Saved 12899 cleaned comments
